# EvalRX on a Colab TPU: build the image (run once)

[Open in Colab](https://colab.research.google.com/github/evalvitals/evalrx/blob/ruinan/examples/colab/tpu/setup.ipynb)

This notebook makes the prebuilt image that [`launch.ipynb`](https://github.com/evalvitals/evalrx/blob/ruinan/examples/colab/tpu/launch.ipynb) restores. Run it once, on a Colab **TPU** runtime with internet access (public Colab works), and save the result where your launch runtimes can read it. It writes:

- `evalrx-colab-tpu-image-<commit>.tar` and `.sha256`, about 5 GB: the installed environment (Python 3.12, the packages in `tools/colab/lock/tpu.txt`, `agy`), the EvalRX source and the two benchmark datasets.
- `evalrx-colab-weights-tpu-gemma-4-e2b.tar` and `.sha256`, 18 GB: the Gemma 4 E2B checkpoint and tokenizer, for runtimes that cannot read `gs://gemma-data`. It does not depend on the EvalRX revision; build it once and keep it.

Rebuild the image to move to newer EvalRX code, or when a new TPU generation needs a newer libtpu than the image has.

## 1. Where to save the image

A Drive path mounts Drive when the build cell runs. For a `gs://` destination on a TPU runtime, first run `from google.colab import auth; auth.authenticate_user()`.

In [ ]:
import os
# Where to save the image: a Drive folder (/content/drive/MyDrive/evalrx), gs://bucket/prefix, or a local directory.
EVALRX_OUT = os.environ.get('EVALRX_OUT', '/content/drive/MyDrive/evalrx')
EVALRX_REF = os.environ.get('EVALRX_REF', 'ruinan')  # EvalRX source revision (branch, tag or commit)
EVALRX_BUNDLE = os.environ.get('EVALRX_BUNDLE', '')  # optional: an offline bundle (build_bundle.sh) as the source
EVALRX_GIT_URL = os.environ.get('EVALRX_GIT_URL', 'https://github.com/evalvitals/evalrx.git')
BUILD_WEIGHTS = os.environ.get('EVALRX_BUILD_WEIGHTS', '1') == '1'  # skip once the weights tar exists

## 2. Install and pack

`tools/colab/bootstrap.sh` installs the locked environment into `/content/evalrx-env` (a few minutes). `tools/colab/build_image.sh` freezes the datasets, checks the environment and writes the image. `tools/colab/build_weights.sh` downloads and packs the weights (about 15 minutes).

In [ ]:
import shutil, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

CONTENT = Path('/content')
repo, env, work = CONTENT / 'evalrx', CONTENT / 'evalrx-env', CONTENT / 'evalrx-image-out'
assert not env.exists(), 'run this notebook on a fresh runtime: /content/evalrx-env already exists'
if EVALRX_OUT.startswith('/content/drive/') and not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')

def run_visible(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

def open_source(src):
    """A binary stream of a local or Drive path, an https:// URL, or a gs:// object."""
    if src.startswith(('http://', 'https://')):
        return urllib.request.urlopen(src)
    if src.startswith('gs://'):
        tool = shutil.which('gcloud') or shutil.which('gsutil')
        if tool:
            cmd = [tool, 'storage', 'cat', src] if tool.endswith('gcloud') else [tool, 'cat', src]
            return subprocess.Popen(cmd, stdout=subprocess.PIPE).stdout
        # No Cloud SDK (e.g. Colab TPU runtimes): the kernel's GCS client, with your
        # credentials (google.colab.auth.authenticate_user()), else anonymous.
        from google.cloud import storage
        bucket, _, name = src[len('gs://'):].partition('/')
        try:
            blob = storage.Client(project=os.environ.get('GOOGLE_CLOUD_PROJECT', '_')).bucket(bucket).blob(name)
            blob.reload()
        except Exception:
            blob = storage.Client.create_anonymous_client().bucket(bucket).blob(name)
        return blob.open('rb')
    return open(src, 'rb')

if EVALRX_BUNDLE:
    with open_source(EVALRX_BUNDLE) as stream, tarfile.open(fileobj=stream, mode='r|') as archive:
        unpack = Path(tempfile.mkdtemp(dir=CONTENT))
        archive.extractall(unpack, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
    (unpack / 'evalrx').rename(repo)
else:
    run_visible(['git', 'init', '-q', str(repo)])
    run_visible(['git', '-C', str(repo), 'fetch', '-q', '--depth', '1',
                 EVALRX_GIT_URL, EVALRX_REF])
    run_visible(['git', '-C', str(repo), 'checkout', '-q', 'FETCH_HEAD'])
# Install (the only step that downloads packages), then pack the installed result.
run_visible(['bash', str(repo / 'tools/colab/bootstrap.sh'), 'tpu'])
run_visible(['bash', str(repo / 'tools/colab/build_image.sh'), 'tpu', str(work)])
if BUILD_WEIGHTS:
    run_visible(['bash', str(repo / 'tools/colab/build_weights.sh'), 'tpu', str(work)])

## 3. Save

Copies the tars and their `.sha256` files to `EVALRX_OUT` and prints the values for `launch.ipynb`.

In [ ]:
files = sorted(p for p in work.iterdir() if p.is_file())
if EVALRX_OUT.startswith('gs://'):
    tool = shutil.which('gcloud') or shutil.which('gsutil')
    if tool:
        run_visible(([tool, 'storage', 'cp'] if tool.endswith('gcloud') else [tool, 'cp'])
                    + [str(p) for p in files] + [EVALRX_OUT.rstrip('/') + '/'])
    else:  # Colab TPU runtimes have no Cloud SDK; needs google.colab.auth.authenticate_user()
        from google.cloud import storage
        bucket, _, prefix = EVALRX_OUT[len('gs://'):].partition('/')
        client = storage.Client(project=os.environ.get('GOOGLE_CLOUD_PROJECT', '_'))
        for p in files:
            client.bucket(bucket).blob(f"{prefix.strip('/')}/{p.name}".lstrip('/')).upload_from_filename(str(p))
else:
    Path(EVALRX_OUT).mkdir(parents=True, exist_ok=True)
    for p in files:
        print('copying', p.name, flush=True)
        shutil.copy(p, Path(EVALRX_OUT) / p.name)
for p in files:
    if p.suffix == '.sha256':
        print(p.read_text().strip())
print('\nIn launch.ipynb set:')
for p in files:
    if p.suffix == '.tar':
        var = 'EVALRX_WEIGHTS' if 'weights' in p.name else 'EVALRX_IMAGE'
        print(f"  {var} = '{EVALRX_OUT.rstrip('/')}/{p.name}'")